In [2]:
#ARC Program functions
#This cell generates the Hamiltonian, solves it, and exports the data
from arc import *
import numpy as np
from scipy.optimize import curve_fit
%matplotlib inline
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

def state_to_string(n,l,j,m):
    if l == 0:
        lstring = "s"
    if l == 1:
        lstring = "p"
    if l == 2:
        lstring = "d"
    return str(int(n)) + "$" + lstring + "_{" + str(j) + "},$ "+ "m=" + str(m)

def arc_calculate(a1,n1, a2,n2):
    if a1 == "Cs":
        atom1 = Caesium()
    elif a1 == "Rb":
        atom1 = Rubidium()
    # n1 = 61
    l1 = 0
    j1 = 0.5
    m1 = 0.5
    if a2 == "Cs":
        atom2 = Caesium()
    elif a2 == "Rb":
        atom2 = Rubidium()
    # n2 = 61
    l2 = 0
    j2 = 0.5
    m2 = 0.5
    

    Bz = 0.0001# #Tesla
    
    rmin = 1.5
    rmax = 6
    
    theta = 0 # Polar Angle [0-pi]
    phi = 0 # Azimuthal Angle [0-2pi]
    dn = 5 # Range of n to consider (n0-dn:n0+dn)
    dl = 5 # Range of l values
    deltaMax = 25e9  # Max pair-state energy difference [Hz]
    
    nEig = 250  # Number of eigenstates to extract
    
    
    calc = PairStateInteractions(atom1, n1, l1, j1, n2, l2, j2, m1, m2, s=0.5, s2=0.5, atom2=atom2)
    r = np.linspace(rmin, rmax, 100)
    # Generate pair-state interaction Hamiltonian
    calc.defineBasis(theta, phi, dn, dl, deltaMax, Bz = Bz, progressOutput=True, debugOutput=False)
    # Diagonalize the Hamiltonian
    calc.diagonalise(r, nEig, progressOutput=True)
    # Plot
    calc.plotLevelDiagram()
    calc.ax.set_xlim(rmin, rmax)
    calc.ax.set_ylim(-0.06, 0.06)
    calc.showPlot()


    calc.exportData("saved-calcs\\" + str(n1) + str(n2), exportFormat='csv')
    separations_file = "saved-calcs\\" + str(n1) + str(n2) + "_r.csv"
    datapoints_file = "saved-calcs\\" + str(n1)+ str(n2) + "_energyLevels.csv"
    highlights_file = "saved-calcs\\" + str(n1) + str(n2) + "_highlight.csv"
    
    separations = np.genfromtxt(separations_file, delimiter=',')
    datapoints = np.genfromtxt(datapoints_file, delimiter=',') #* 1000 #Units are now in MHz
    highlights = np.genfromtxt(highlights_file, delimiter=',')
    # print(datapoints[0].shape)
    
    highest_overlap_points = []
    
    fig, ax = plt.subplots(figsize=(8, 6))
    norm = mcolors.LogNorm(vmin=0.001, vmax=1)
    colors = ["whitesmoke", "orange", "darkred"]
    cmap = mcolors.LinearSegmentedColormap.from_list("custom_colormap", colors, N=256)
    
    for i in range(len(separations)):#for each r
        xplot = np.ones(len(datapoints[i])) * separations[i]
        #Sort the data so that the most overlap data is on top.
        sorted_pairs = sorted(zip(highlights[i], datapoints[i]))  # Sort by highlights
        sorted_highlights, sorted_ydata = zip(*sorted_pairs)  # Unzip into separate lists
        # Convert back to lists
        sorted_ydata = list(sorted_ydata)
        sorted_highlights = list(sorted_highlights) 
        # print(xplot)
        # print(sorted_ydata)
        highest_overlap_points.append(sorted_ydata[-1])  # Appends the strongest eigenvalue for each x point
        plt.scatter(xplot, sorted_ydata,c=sorted_highlights, cmap=cmap, norm = norm, s=8)
    
    plt.ylim((-0.1,0.1))
    plt.xlim(np.min(separations),np.max(separations))
    plt.axhline(y=0.0, color='black', linestyle='-')
    cbar = plt.colorbar()
    cbar.set_label('Overlap', fontsize = 16)
    plt.xlabel("$r$ Inter-Nuclear Distance ($\\mu$m)",fontsize=20)
    plt.ylabel("Pair State Energy (GHz)",fontsize=20)
    a1label = a1 +": " + state_to_string(n1,l1,j1,m1)
    a2label = a2 +": " + state_to_string(n2,l2,j2,m2)
    calc_data = str(theta * 180 / np.pi) + "$\\degree,$ $B_z = $ " + str(Bz * 1e4) + " G"
    plt.text(4,0.12,a1label)
    plt.text(4,0.1,a2label)
    plt.text(4,0.08,calc_data)
    plt.savefig(str(n1) + ".png")
    plt.show()
# def arc_plot(n):
#     separations_file = "output_r.csv"
#     datapoints_file = "output_energyLevels.csv"
#     highlights_file = "output_highlight.csv"
    
#     separations = np.genfromtxt(separations_file, delimiter=',')
#     datapoints = np.genfromtxt(datapoints_file, delimiter=',') #* 1000 #Units are now in MHz
#     highlights = np.genfromtxt(highlights_file, delimiter=',')
    
#     # print(datapoints[0].shape)
    
#     highest_overlap_points = []
    
#     fig, ax = plt.subplots(figsize=(8, 6))
#     norm = mcolors.LogNorm(vmin=0.001, vmax=1)
#     colors = ["whitesmoke", "orange", "darkred"]
#     cmap = mcolors.LinearSegmentedColormap.from_list("custom_colormap", colors, N=256)
    
#     for i in range(len(separations)):#for each r
#         xplot = np.ones(len(datapoints[i])) * separations[i]
#         #Sort the data so that the most overlap data is on top.
#         sorted_pairs = sorted(zip(highlights[i], datapoints[i]))  # Sort by highlights
#         sorted_highlights, sorted_ydata = zip(*sorted_pairs)  # Unzip into separate lists
#         # Convert back to lists
#         sorted_ydata = list(sorted_ydata)
#         sorted_highlights = list(sorted_highlights) 
#         # print(xplot)
#         # print(sorted_ydata)
#         highest_overlap_points.append(sorted_ydata[-1])  # Appends the strongest eigenvalue for each x point
#         plt.scatter(xplot, sorted_ydata,c=sorted_highlights, cmap=cmap, norm = norm, s=8)
    
#     plt.ylim((-0.150,0.150))
#     plt.xlim(np.min(separations),np.max(separations))
#     plt.axhline(y=0.0, color='black', linestyle='-')
#     cbar = plt.colorbar()
#     cbar.set_label('Overlap', fontsize = 16)
#     plt.xlabel("$r$ Inter-Nuclear Distance ($\\mu$m)",fontsize=20)
#     plt.ylabel("Pair State Energy (GHz)",fontsize=20)
#     plotlabel = atomstring(atom1) + str(int(n)) + "$p_{3/2}$, $m=+0.5$, $B=1.$0 G, $90 \\degree$"
#     plt.text(4,0.4,plotlabel)
#     plt.savefig("90-deg-m-0.5-4.5G" + str(n) + ".png")


ModuleNotFoundError: No module named 'arc'

In [3]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
Created on Fri Aug  1 14:50:23 2025

@author: yeddie
"""

#ARC Program functions
#This cell generates the Hamiltonian, solves it, and exports the data
from arc import *
import numpy as np
from scipy.optimize import curve_fit
from scipy.sparse.linalg import eigsh

%matplotlib inline
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

def top_overlap_eigenpairs(calc, r_um, ref_indices, k_return=10, k_eigs=250, sigma=0.0):
    H = hamiltonian_at(calc, r_um)
    k_eigs = min(k_eigs, H.shape[0]-1)          # ARPACK requirement
    evals, evecs = eigsh(H, k=k_eigs, sigma=sigma, which="LM", tol=1e-6)  # evals in GHz
    # Overlap weight with a set of basis indices (sum |coeff|^2)
    if len(ref_indices) == 0:
        raise ValueError("Reference basis state not found in calc.basisStates.")
        
    weights = np.sum(np.abs(evecs[ref_indices, :])**2, axis=0)            # shape (k_eigs,)
    # Rank by overlap
    order = np.argsort(-weights)[:k_return]
    top = []
    for j in order:
        w = weights[j]
        ev = evals[j]
        vec = evecs[:, j]  # eigenvector in the calc.basisStates order
        top.append((w, ev, vec))
    return top, evals, evecs

def hamiltonian_at(calc, r_um: float):
    H = calc.matDiagonal.copy()        # sparse, units: GHz
    r_si = r_um * 1e-6                 # meters
    rpow = r_si**3
    for MR in calc.matR:               # 1/R^3, 1/R^4, 1/R^5, ...
        H = H + MR / rpow
        rpow *= r_si
    return H
    
def find_basis_indices(calc, n1,l1,j1,m1, n2,l2,j2,m2):
    idxs = []
    for k, st in enumerate(calc.basisStates):
        # ARC basis entries typically have these attributes; keep checks robust:
        a = getattr(st, 'atom1', None)
        b = getattr(st, 'atom2', None)
        if a and b:
            ok1 = (a.n == n1 and a.l == l1 and a.j == j1 and a.mj == m1)
            ok2 = (b.n == n2 and b.l == l2 and b.j == j2 and b.mj == m2)
            if ok1 and ok2:
                idxs.append(k)
    return idxs
    
def state_to_string(n,l,j,m):
    if l == 0:
        lstring = "s"
    if l == 1:
        lstring = "p"
    if l == 2:
        lstring = "d"
    return str(int(n)) + "$" + lstring + "_{" + str(j) + "},$ "+ "m=" + str(m)

def arc_calculate(a1,n1,a2,n2):
    if a1 == "Cs":
        atom1 = Caesium()
    elif a1 == "Rb":
        atom1 = Rubidium87()
    # n1 = 61
    l1 = 0
    j1 = 0.5
    m1 = 0.5
    if a2 == "Cs":
        atom2 = Caesium()
    elif a2 == "Rb":
        atom2 = Rubidium87()
    # n2 = 61
    l2 = 0
    j2 = 0.5
    m2 = 0.5
    

    Bz = 0.0001# #Tesla
    
    rmin = 1.5
    rmax = 6
    
    theta = 0 # Polar Angle [0-pi]
    phi = 0 # Azimuthal Angle [0-2pi]
    dn = 5 # Range of n to consider (n0-dn:n0+dn)
    dl = 5 # Range of l values
    deltaMax = 25e9  # Max pair-state energy difference [Hz]
    
    nEig = 250  # Number of eigenstates to extract
    
    
    calc = PairStateInteractions(atom1, n1, l1, j1, n2, l2, j2, m1, m2, s=0.5, s2=0.5, atom2=atom2)
    r = np.linspace(rmin, rmax, 100)
    # Generate pair-state interaction Hamiltonian
    calc.defineBasis(theta, phi, dn, dl, deltaMax, Bz = Bz, progressOutput=True, debugOutput=False)
    # Diagonalize the Hamiltonian
    calc.diagonalise(r, nEig, progressOutput=True)
    # Plot
    #calc.plotLevelDiagram()
    #calc.ax.set_xlim(rmin, rmax)
    #calc.ax.set_ylim(-0.06, 0.06)
    #calc.showPlot()
    
    r_target = 2.5  # µm
    H = hamiltonian_at(calc, r_target)
    ref_idxs = find_basis_indices(calc, n1,l1,j1,m1, n2,l2,j2,m2)
    print("Reference basis indices:", ref_idxs)  # should be 1 (often) or more if degenerate

    top10, evals, evecs = top_overlap_eigenpairs(calc, r_target, ref_idxs, k_return=10, k_eigs=250, sigma=0.0)

    for rank, (w, ev, vec) in enumerate(top10, start=1):
        print(f"{rank:2d}) overlap={w:.4f},  E={ev:.6f} GHz,  ||vec||={np.linalg.norm(vec):.3f}")
    
    """
    # When no need for data 
    calc.exportData("saved-calcs\\" + str(n1) + str(n2), exportFormat='csv')
    separations_file = "saved-calcs\\" + str(n1) + str(n2) + "_r.csv"
    datapoints_file = "saved-calcs\\" + str(n1)+ str(n2) + "_energyLevels.csv"
    highlights_file = "saved-calcs\\" + str(n1) + str(n2) + "_highlight.csv"
    
    separations = np.genfromtxt(separations_file, delimiter=',')
    datapoints = np.genfromtxt(datapoints_file, delimiter=',') #* 1000 #Units are now in MHz
    highlights = np.genfromtxt(highlights_file, delimiter=',')
    # print(datapoints[0].shape)
    
    highest_overlap_points = []
    
    fig, ax = plt.subplots(figsize=(8, 6))
    norm = mcolors.LogNorm(vmin=0.001, vmax=1)
    colors = ["whitesmoke", "orange", "darkred"]
    cmap = mcolors.LinearSegmentedColormap.from_list("custom_colormap", colors, N=256)
  
    """
"""
# No need for the plotting in this case
    for i in range(len(separations)):#for each r
        xplot = np.ones(len(datapoints[i])) * separations[i]
        #Sort the data so that the most overlap data is on top.
        sorted_pairs = sorted(zip(highlights[i], datapoints[i]))  # Sort by highlights
        sorted_highlights, sorted_ydata = zip(*sorted_pairs)  # Unzip into separate lists
        # Convert back to lists
        sorted_ydata = list(sorted_ydata)
        sorted_highlights = list(sorted_highlights) 
        # print(xplot)
        # print(sorted_ydata)
        highest_overlap_points.append(sorted_ydata[-1])  # Appends the strongest eigenvalue for each x point
        plt.scatter(xplot, sorted_ydata,c=sorted_highlights, cmap=cmap, norm = norm, s=8)
    
    plt.ylim((-0.1,0.1))
    plt.xlim(np.min(separations),np.max(separations))
    plt.axhline(y=0.0, color='black', linestyle='-')
    cbar = plt.colorbar()
    cbar.set_label('Overlap', fontsize = 16)
    plt.xlabel("$r$ Inter-Nuclear Distance ($\\mu$m)",fontsize=20)
    plt.ylabel("Pair State Energy (GHz)",fontsize=20)
    a1label = a1 +": " + state_to_string(n1,l1,j1,m1)
    a2label = a2 +": " + state_to_string(n2,l2,j2,m2)
    calc_data = str(theta * 180 / np.pi) + "$\\degree,$ $B_z = $ " + str(Bz * 1e4) + " G"
    plt.text(4,0.12,a1label)
    plt.text(4,0.1,a2label)
    plt.text(4,0.08,calc_data)
    plt.savefig(str(n1) + ".png")
    plt.show()
"""    



arc_calculate("Cs", 41, "Rb", 32)






ModuleNotFoundError: No module named 'arc'